# 28 · The skin degradome — which TME subtype makes the proteases

`24_subclone_tcr_signaling` scores one `protease` program on malignant subclones (`subclone_helpers.py:511`, 11
genes, `sc.tl.score_genes` per cell) and renders it as a single column of
`figures/subclone_v3_programs.png`. It is never tested; the per-gene dot plot is dead code
(`panels` at `24_subclone_tcr_signaling` cell 11 is computed and never consumed); and the one time the panel *was*
tested gene by gene (`old/23_subclonal_evolution.ipynb` cell 17) the sole up-call was
**TIMP1** — the inhibitor. The read "ECM proteolysis in MF skin is not tumour-intrinsic" is
supported by that, and was never followed by the obvious next question.

**This notebook asks who does make it.** `22_myeloid_fibro_reannotation` produced 13 myeloid and fibroblast subtypes
that did not exist when the `24_subclone_tcr_signaling` panel was written; `27_ccc_subtype` uses them for ligand–receptor work
and never for expression. Here every protease, ECM-degrading enzyme and endogenous inhibitor
is attributed to a producing cell type, at that subtype resolution, with donor-level spread.

## What is claimed, and what is only counted

| | levels | role |
|---|---|---|
| **claim** | 13 `22_myeloid_fibro_reannotation` subtypes + `CD4_malignant` | statements are made about these |
| **context** | Keratinocyte, CD4_reactive, CD4_unassessed, CD8, Tregs, B, Plasma, Vascular, Mast, Melanocyte | denominator only — no claim, no test |

"cDC supplies 40% of the skin's MMP12" is meaningless without a complete denominator, so
every skin level is carried. Nothing is asserted about the context levels. **Mast cells will
own several genes; that is the positive control (§2b), not a finding.**

## The power constraint, settled before any code ran

Skin has **9 HC donors**, from 3 studies. At the 25-cell gate the CTCL-vs-HC contrast is
defined for **6 of 13** subtypes — cDC (8 HC donors), F_reticular (8), F_papillary (8),
F_mesenchymal (7), Mac_FOLR2 (5), F_apCAF (4) — and for **none** of DC_LAMP3, Mac_infl,
Mono_moDC, Mac_SPP1_TREM2, LC, pDC, F_inflammatory. Those seven are not null: they are absent
from healthy skin at the gate, which is an abundance observation (§5) and never a fold change.
`chennareddy2025` and `gaydosik2019` each carry both arms, so the contrast runs **within
study**; a pooled fit would put disease and study in the same design column and is kept only
as a labelled sensitivity arm.

## Sections

§0 build gate, roster, coverage · §1 panel coverage · §2 source attribution (+ controls) ·
§3 spillover · §4 CTCL vs HC · §5 the levels with no healthy comparator · §6 figures + log.

Config is `degradome_data.py`, every function is in `degradome_helpers.py`. Nothing is
defined here. The object comes from `jobs/run_degradome_build.sh` — one bsub pass over the
48 GB source atlas, because ADAM10 and ADAM17 are in `24_subclone_tcr_signaling`'s panel and are not among the 10k
HVGs of `joint_mrvi_input_skin.h5ad`.

Every section writes `tables/deg42_*.csv` and appends a self-describing block to
`tables/deg42_run_log.md` that reads correctly pasted into a chat with no access to this
notebook.

**Kernel: `neural_nmf_env`.**

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# The repo is not necessarily an ancestor of cwd (a kernel started from $HOME sees it as a
# descendant), so walking up is a convenience, not the definition. FALLBACK is authoritative
# and matches degradome_data.NB_DIR.
FALLBACK = Path("/home/projects/nyosef/zvise/MF")


def _resolve_nb_dir():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if cand.name == "MF" and (cand / "data").exists():
            return cand
        if (cand / "notebooks" / "MF" / "data").exists():
            return cand / "notebooks" / "MF"
    if (FALLBACK / "data").exists():
        return FALLBACK
    raise RuntimeError(f"cannot locate the MF tree from {p} and {FALLBACK} is absent")


NB_DIR = _resolve_nb_dir()
sys.path.insert(0, str(NB_DIR / "helpers"))

# Reloaded explicitly, not merely imported: these two modules are edited while the notebook
# is open, and a kernel that imported them at session start will happily keep serving the old
# code -- which shows up as a TypeError on a parameter that plainly exists on disk.
import importlib                  # noqa: E402
import degradome_data as cfg      # noqa: E402
import degradome_helpers as D     # noqa: E402

cfg = importlib.reload(cfg)
D = importlib.reload(D)
D.cfg = cfg                       # the helper module caches its own reference to the config

# Slow results (the three DESeq2 fits, the shuffle null, the equivalence gate) are cached
# under data/degradome/cache and stamped with the built object's mtime. Set True -- or call
# D.clear_cache() -- to force a recompute; rebuilding the h5ad invalidates them anyway.
RECOMPUTE = False

SEED = cfg.SEED
np.random.seed(SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
mpl.rcParams["figure.dpi"] = 110
mpl.rcParams["savefig.bbox"] = "tight"
cfg.TAB_DIR.mkdir(parents=True, exist_ok=True)
cfg.FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"NB_DIR    {NB_DIR}")
print(f"object    {cfg.DEG_ADATA}")
print(f"panel     {len(cfg.panel_genes(False))} degradome + {len(cfg.CONTEXT_GENES)} identity")
print(f"claim     {len(cfg.CLAIM_LEVELS)} levels | context {len(cfg.CONTEXT_LEVELS)}")

## §0 · The build gate, the roster, the coverage

Three things are established before a single number is read: the panel is complete, the
object's expression values are what the build claims, and the roster reconciles cell-for-cell
against the `22_myeloid_fibro_reannotation` sidecar.

The equivalence gate is **blocking**. `stream_lognorm_subset` sums the library size over all
40,821 genes *before* subsetting to the 156 panel columns. Doing it the other way round
rescales every cell by (panel counts / total counts) — about 40× here — and, worse, by a
factor that varies with cell type, which is exactly the axis this notebook measures. The
result would look entirely plausible and be wrong everywhere. The test set is 3 donors at
full gene width; every panel value must match to `<1e-6`.

In [ ]:
adata = D.load_degradome_adata()

# The gate reads a 95,233 x 40,821 test set, so its verdict is cached rather than its data.
gate = D.cached("equivalence",
                lambda: {"max_abs_diff": D.assert_build_equivalence(adata)},
                force=RECOMPUTE)
print(f"equivalence gate: max abs diff {gate['max_abs_diff']:.3e}")

In [ ]:
cov = D.coverage_table(adata.obs, out=cfg.tab("coverage"))
testable = D.assert_power_statement(cov)

## §1 · Panel coverage

Detection fraction per (gene, level) over eligible donors. Genes never detected above 1% of
cells in any level are dropped from the attribution — a share computed from three counts is
noise, not attribution — and the drop list is printed, because silent truncation reads as
"we looked at everything" when we did not.

The `(level, donor)` cube is built here by one indicator matmul. **Donor, not sample, is the
replicate unit**: several biopsies from one patient are not independent observations of that
patient's stroma.

In [ ]:
cube = D.DegradomeCube(adata)
panel_cov, dropped = D.panel_coverage(cube, out=cfg.tab("panel_coverage"))
GENES = [g for g in cfg.panel_genes(include_context=False) if g in set(panel_cov.index[panel_cov["keep"]])]
print(f"\n{len(GENES)} degradome genes carried into the attribution")

## §2 · Source attribution — the primary claim

Per donor, per gene: what fraction of that donor's total skin transcript output for the gene
came from each level. Two shares, answering different questions.

- **`share_raw`** — of every transcript of gene *g* captured in this donor's skin, how much
  came from level *l*. Abundance is part of the answer: a fibroblast population ten times
  larger genuinely supplies ten times the collagenase. **This is the attribution.**
- **`share_cpm`** — the same after normalising each level by its own full-gene library, so
  every level contributes per unit of transcriptome. Abundance divided out; this is intensity.

A gene owned on `share_raw` but not on `share_cpm` is supplied by a level because there is a
lot of it, not because each cell makes much. Both are reported.

A level **owns** a gene when it is the top producer in ≥⅔ of the donors where the share is
computable, over ≥5 donors. Genes clearing neither get `owner = None` — shared production is
a real answer, not a missing one.

In [ ]:
attr = D.attribution(cube, genes=GENES, out=cfg.tab("attribution"))
own = D.call_owners(attr, out=cfg.tab("gene_owner"))

### §2b · Controls

**Positive.** The attribution has a known right answer for a handful of genes: mast tryptases
must come from mast cells, granzymes from CD8, the kallikrein cascade and SPINK5 from
keratinocytes, COL1A1 from fibroblasts. These are not results — they are the only way to know
the share arithmetic, the `cell_id` join and the library sizes are right before any actual
claim is read. A miss hard-fails.

**Negative.** The level labels are permuted within donor, which preserves every level's size
and every donor's composition and destroys only the cell-to-level link. The resulting win
fraction is what this many donors and this much sparsity produce by chance. An owner call
that does not clear that ceiling is not an attribution.

In [ ]:
ctrl = D.cached("positive_controls",
                lambda: D.assert_positive_controls(adata, strict=False),
                force=RECOMPUTE)
D.assert_controls_pass(ctrl)      # gated even when cached
ctrl.to_csv(cfg.tab("positive_controls"), index=False)

In [ ]:
null = D.cached("shuffle_null",
                lambda: D.shuffle_null(adata, cube, genes=GENES),
                force=RECOMPUTE)
own = D.against_null(own, null, out=cfg.tab("gene_owner"))

### §2c · The answer to `24_subclone_tcr_signaling`'s question

The same arithmetic, restricted to the twelve genes `24_subclone_tcr_signaling` actually scored. If the tumour clone
is not the source, this table names what is.

In [ ]:
D.assert_nb31_panel()

# (a) subtype roster -- "which macrophage state", the resolution nb10c bought us
nb31 = own[own["gene"].isin(cfg.NB31_PANEL)].set_index("gene").reindex(cfg.NB31_PANEL)
mal = attr[(attr[cfg.GROUPBY] == cfg.CD4_MALIGNANT) & (attr["gene"].isin(cfg.NB31_PANEL))]
nb31["CD4_malignant_share"] = mal.set_index("gene")["share_raw_med"].reindex(cfg.NB31_PANEL).round(4)
nb31 = nb31.reset_index()
nb31.to_csv(cfg.tab("nb31_panel_owner_subtype"), index=False)
print("=== subtype roster ===")
print(nb31[["gene", "family", "owner", "owner_win_frac", "owner_share_med",
            "CD4_malignant_share", "n_donors_abstained"]].to_string(index=False))

# (b) lineage roster -- the granularity that can answer "tumour or stroma". A gene with no
# single-subtype owner is not a gene with no source; production split across five fibroblast
# states still has a lineage.
attr_lin, own_lin, cube_lin = D.lineage_attribution(adata, cfg.NB31_PANEL, verbose=False)
mal_lin = attr_lin[attr_lin[cfg.GROUPBY] == cfg.CD4_MALIGNANT]
lin = own_lin.set_index("gene").reindex(cfg.NB31_PANEL)
lin["CD4_malignant_share"] = mal_lin.set_index("gene")["share_raw_med"].reindex(cfg.NB31_PANEL).round(4)
lin = lin.reset_index()
lin.to_csv(cfg.tab("nb31_panel_owner_lineage"), index=False)
print("\n=== lineage roster (TME subtypes collapsed to Myeloid / Fibroblast) ===")
print(lin[["gene", "owner", "owner_win_frac", "owner_share_med",
           "CD4_malignant_share"]].to_string(index=False))

n_mal = int((lin["owner"] == cfg.CD4_MALIGNANT).sum())
print(f"\nof {len(lin)} nb31 panel genes, {n_mal} are owned by CD4_malignant at lineage "
      f"granularity; median malignant share = {lin['CD4_malignant_share'].median():.4f}")

## §3 · Spillover control

Ambient mRNA and doublets put a cell type's transcripts into every other cell type in the same
droplet suspension, in proportion to how abundant that cell type is. So for a level that does
**not** own a gene, a per-cell intensity (abundance already divided out) that still rises with
the owner's donor cell fraction is contamination, not expression.

Flagged, never dropped: an inflamed donor really does have both more macrophages and more
macrophage-adjacent signalling, so a positive correlation is suggestive and not proof. The
second guard is detection — ambient raises the mean at low detection, so a high share with a
low `det_frac` is the shape to distrust.

In [ ]:
spill = D.spillover_flags(cube, own, genes=GENES, out=cfg.tab("spillover"))
own = D.owner_spillover_note(own, cube)
own.to_csv(cfg.tab("gene_owner"), index=False)
low = own[own["owner"].notna() & own["owner_low_detection"]]
print(f"\n{len(low)} owner calls rest on a high share at < 10% detection in the owner:")
print(low[["gene", "owner", "owner_share_med", "owner_det_frac"]].to_string(index=False))

### §3b · Protease : inhibitor balance

The readout the `24_subclone_tcr_signaling` comment promised — *"read the protease:TIMP1 ratio in the dot plot"* —
and never rendered, computed on families rather than one inhibitor, in CPM on each level's own
full-gene library, so it is net degradative tone per unit of transcriptome and does not move
with cell number.

In [ ]:
bal = D.balance_table(cube, out=cfg.tab("balance"))

## §4 · CTCL vs healthy skin, within level

Pseudobulk DESeq2 on the **full-gene** cube (so size factors and dispersions are estimated on
the whole transcriptome, not on 156 columns), collapsed to donor, run separately inside each
level, design `~ study + disease_grp`. BH is applied within level across the panel genes,
because the panel is the hypothesis.

Restricted to `chennareddy2025` + `gaydosik2019` — the only two skin studies contributing
donors to both arms. Five of seven are CTCL-only, so a pooled fit reports batch as biology;
it is run below as a labelled sensitivity arm and is not a headline.

In [ ]:
agg, dmeta = D.donor_pseudobulk()
de = D.cached("contrast_disease",
              lambda: D.disease_contrast(agg, dmeta, levels=testable, arm="within_study"),
              force=RECOMPUTE)
de_pooled = D.cached("contrast_disease_pooled",
                     lambda: D.disease_contrast(agg, dmeta, levels=testable,
                                                arm="pooled_sensitivity"),
                     force=RECOMPUTE)

# epidermis vs dermis: paired within donor, so the pseudobulk must keep the layer split.
# Needs no healthy arm, which is why it reaches DC_LAMP3 and Mac_infl.
de_layer = D.cached("contrast_layer",
                    lambda: D.layer_contrast(*D.donor_pseudobulk(extra_keys=(cfg.LAYER_KEY,))),
                    force=RECOMPUTE)

# advanced vs early: li2024 is the only study containing both stages, so the contrast is run
# inside it and is free of study confounding by construction.
de_stage = D.cached("contrast_stage", lambda: D.stage_contrast(agg, dmeta), force=RECOMPUTE)

de_all = pd.concat([de, de_pooled, de_layer, de_stage], ignore_index=True)
de_all.to_csv(cfg.tab("contrasts"), index=False)

sig = de[(de["padj_panel"] < 0.05) & (de["log2FoldChange"].abs() >= 1.0)]
print(f"\nCTCL vs HC within study: {len(sig)} calls at panel FDR < 0.05, |log2FC| >= 1")
print(sig.sort_values("padj_panel")[["gene", "family", cfg.GROUPBY, "log2FoldChange",
                                     "padj_panel"]].head(20).to_string(index=False))
for name, d in [("epidermis vs dermis", de_layer), ("advanced vs early", de_stage)]:
    s = d[(d["padj_panel"] < 0.05) & (d["log2FoldChange"].abs() >= 1.0)] if len(d) else d
    print(f"\n{name}: {len(s)} calls; levels {sorted(set(d[cfg.GROUPBY])) if len(d) else '(none)'}")
    if len(s):
        print(s.sort_values("padj_panel")[["gene", "family", cfg.GROUPBY, "log2FoldChange",
                                           "padj_panel"]].head(15).to_string(index=False))

In [ ]:
# Agreement between the two arms -- a within-study call that flips sign when pooled is a
# study effect wearing a disease label.
if len(de) and len(de_pooled):
    j = de.merge(de_pooled, on=["gene", cfg.GROUPBY], suffixes=("_within", "_pooled"))
    j["sign_agrees"] = np.sign(j["log2FoldChange_within"]) == np.sign(j["log2FoldChange_pooled"])
    s = j[j["padj_panel_within"] < 0.05]
    print(f"of {len(s)} within-study calls, {int(s['sign_agrees'].sum())} keep their sign when "
          f"every study is pooled")

## §5 · The seven levels with no healthy comparator

DC_LAMP3, Mac_infl, Mono_moDC, Mac_SPP1_TREM2, LC, pDC and F_inflammatory clear the claim gate
in lesional skin and have **zero** HC donors at that gate. Their degradome output is reported
here as an abundance observation with no fold change attached. Writing them as "not
significant" would be the single most misleading thing this notebook could do.

In [ ]:
lesion = D.lesion_restricted_table(cov, cube, out=cfg.tab("lesion_restricted"))

## §6 · Figures

Four figures, one question each: who supplies the `24_subclone_tcr_signaling` panel, the protease:inhibitor balance
per level, and the three contrasts. The gene x level heatmap over the whole panel was dropped
-- 112 rows against 24 columns is a lookup table, and `tables/deg42_attribution.csv` is the
lookup table.

In [ ]:
D.plot_nb31_panel_owner(attr, save=cfg.fig("nb31_panel_owner"))

In [ ]:
D.plot_balance(bal, save=cfg.fig("protease_inhibitor_balance"))

In [ ]:
D.plot_volcano(de, levels=cfg.DISEASE_TESTABLE, arm="within_study",
               title="CTCL vs healthy skin", save=cfg.fig("volcano_disease"))
D.plot_volcano(de_layer, title="epidermis vs dermis", save=cfg.fig("volcano_layer"))
D.plot_volcano(de_stage, title=f"advanced vs early ({cfg.STAGE_STUDY})",
               save=cfg.fig("volcano_stage"))

### §6b · The machine-readable summary

One block per section, appended to `tables/deg42_run_log.md`. It has to be readable pasted
into a chat by someone with no access to this notebook, so every number carries its
denominator and every abstention says why.

In [ ]:
called = own[own["owner"].notna()]
mal_owned = called[called["owner"] == cfg.CD4_MALIGNANT]


def _contrast_line(name, d, extra=""):
    if not len(d):
        return f"{name:<26}: no level cleared the design {extra}"
    s = d[(d["padj_panel"] < 0.05) & (d["log2FoldChange"].abs() >= 1.0)]
    return (f"{name:<26}: {len(s)} calls at panel FDR < 0.05, |log2FC| >= 1 across "
            f"{sorted(set(d[cfg.GROUPBY]))} {extra}")


body = "\n".join([
    f"object          : {cfg.DEG_ADATA.name}  {adata.n_obs:,} cells x {adata.n_vars} genes",
    f"donors / samples: {adata.obs[cfg.DONOR_KEY].nunique()} / {adata.obs[cfg.SAMPLE_KEY].nunique()}",
    f"levels          : {len(cfg.CLAIM_LEVELS)} claim, {len(cfg.CONTEXT_LEVELS)} context (denominator only)",
    f"panel           : {len(GENES)} of {len(cfg.panel_genes(False))} degradome genes cleared detection",
    f"dropped genes   : {', '.join(dropped) if dropped else '(none)'}",
    "",
    f"genes with a single-SUBTYPE owner : {len(called)} of {len(own)}",
    f"  owned by CD4_malignant          : {len(mal_owned)}"
    + (f"  ({', '.join(mal_owned['gene'])})" if len(mal_owned) else ""),
    f"  above the shuffle ceiling       : {int((called['above_null'] == True).sum())}",
    f"  resting on < 10% detection      : {int(called['owner_low_detection'].sum())}",
    "",
    "nb31 protease panel at LINEAGE granularity (the resolution that can say tumour vs stroma):",
    lin[["gene", "owner", "owner_win_frac", "owner_share_med",
         "CD4_malignant_share"]].to_string(index=False),
    "",
    f"spillover-flagged (gene, non-owner level) pairs: "
    f"{int(spill['spillover_suspect'].sum()) if len(spill) else 0} of {len(spill)}",
    "",
    "CONTRASTS",
    _contrast_line("CTCL vs HC", de, f"[within {' + '.join(cfg.PAIRED_STUDIES)}]"),
    _contrast_line("epidermis vs dermis", de_layer, "[paired within donor]"),
    _contrast_line("advanced vs early", de_stage, f"[within {cfg.STAGE_STUDY}]"),
    "",
    f"no healthy comparator at the gate (abstained, NOT null): {cfg.DISEASE_UNDEFINED}",
    f"  of these, testable across layers instead: "
    f"{[l for l in cfg.LAYER_TESTABLE if l in cfg.DISEASE_UNDEFINED]}",
])
D.append_run_log("SECTIONS 0-5 -- DEGRADOME ATTRIBUTION", body)

### Outcome

Fill this in from the printed summary once the notebook has run clean: which subtypes own the
MMP, cathepsin and serine arms; whether `CD4_malignant` owns anything at all; and which of the
six testable levels induce their degradome in lesional skin. State the abstentions in the same
breath — seven subtypes have no healthy comparator, and that is a coverage fact about this
atlas, not a biological null.